# Phase 2: Feature Engineering
## Santander Product Recommendation — Next Product to Buy

**Objective:** Transform raw data into powerful features for our prediction model.

**CRITICAL DESIGN RULE:** All product-derived features must use **previous month (lag-1)** data only.
Using current month product ownership = data leakage (the target IS derived from current month).

**Key EDA Insights Driving Feature Design:**
- Payroll + Pensions(Payroll) + Direct Debit are a natural bundle (corr 0.78-0.95)
- Most customers have only 1-2 products → cross-sell potential is the signal
- TOP segment dominates product ownership → segment is a strong feature
- ~20% missing in `renta` (income) → needs careful imputation
- July 2015 structural break in Current Account → needs handling

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = Path('../data/raw/train_ver2.csv')
PROCESSED_PATH = Path('../data/processed')
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

product_cols = [
    'ind_ahor_fin_ult1', 'ind_aval_fin_ult1', 'ind_cco_fin_ult1',
    'ind_cder_fin_ult1', 'ind_cno_fin_ult1', 'ind_ctju_fin_ult1',
    'ind_ctma_fin_ult1', 'ind_ctop_fin_ult1', 'ind_ctpp_fin_ult1',
    'ind_deco_fin_ult1', 'ind_deme_fin_ult1', 'ind_dela_fin_ult1',
    'ind_ecue_fin_ult1', 'ind_fond_fin_ult1', 'ind_hip_fin_ult1',
    'ind_plan_fin_ult1', 'ind_pres_fin_ult1', 'ind_reca_fin_ult1',
    'ind_tjcr_fin_ult1', 'ind_valo_fin_ult1', 'ind_viv_fin_ult1',
    'ind_nomina_ult1', 'ind_nom_pens_ult1', 'ind_recibo_ult1'
]

print('Setup complete.')

Setup complete.


## 1. Data Loading & Initial Cleaning

In [2]:
dtype_dict = {
    'ncodpers': 'int32',
    'ind_empleado': 'category',
    'pais_residencia': 'category',
    'sexo': 'category',
    'ind_nuevo': 'float32',
    'indrel': 'float32',
    'indrel_1mes': 'category',
    'tiprel_1mes': 'category',
    'indresi': 'category',
    'indext': 'category',
    'canal_entrada': 'category',
    'indfall': 'category',
    'nomprov': 'category',
    'ind_actividad_cliente': 'float32',
    'segmento': 'category',
}

df = pd.read_csv(DATA_PATH, dtype=dtype_dict, parse_dates=['fecha_dato', 'fecha_alta'], low_memory=False)
print(f'Loaded: {df.shape}')

Loaded: (13647309, 48)


In [3]:
# --- DROP COLUMNS ---
drop_cols = ['conyuemp', 'ult_fec_cli_1t', 'tipodom']
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
print(f'After dropping useless columns: {df.shape}')

After dropping useless columns: (13647309, 45)


In [4]:
# --- CLEAN INDIVIDUAL COLUMNS ---

df['age'] = pd.to_numeric(df['age'], errors='coerce').clip(18, 100)
df['antiguedad'] = pd.to_numeric(df['antiguedad'], errors='coerce').clip(lower=0)
df['renta'] = pd.to_numeric(df['renta'], errors='coerce')

df['indrel_1mes'] = df['indrel_1mes'].astype(str).str.strip()
df['indrel_1mes'] = df['indrel_1mes'].replace({'1.0': '1', '2.0': '2', '3.0': '3', '4.0': '4', 'P': '5', 'nan': np.nan})

for col in product_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype('int8')

print('Column cleaning complete.')

Column cleaning complete.


In [5]:
# --- IMPUTE MISSING VALUES ---

# renta: impute by province median (FAST vectorized)
renta_medians = df.groupby('nomprov')['renta'].median()
df['renta'] = df['renta'].fillna(df['nomprov'].map(renta_medians))
df['renta'] = df['renta'].fillna(df['renta'].median())

df['sexo'].fillna(df['sexo'].mode()[0], inplace=True)
df['age'].fillna(df['age'].median(), inplace=True)
df['antiguedad'].fillna(0, inplace=True)

for col in ['segmento', 'canal_entrada', 'ind_actividad_cliente', 'ind_nuevo',
            'indrel', 'indresi', 'indext', 'indfall', 'ind_empleado', 'pais_residencia']:
    if col in df.columns and df[col].isnull().any():
        df[col].fillna(df[col].mode()[0], inplace=True)

print(f'Remaining nulls: {df.isnull().sum().sum()}')

Remaining nulls: 1195116


## 2. Target Variable: New Product Additions

In [6]:
df = df.sort_values(['ncodpers', 'fecha_dato']).reset_index(drop=True)

for col in tqdm(product_cols, desc='Computing additions'):
    prev = df.groupby('ncodpers')[col].shift(1)
    df[f'{col}_added'] = ((prev == 0) & (df[col] == 1)).astype('int8')

added_cols = [f'{c}_added' for c in product_cols]

first_month_mask = df.groupby('ncodpers').cumcount() == 0
print(f'First-month rows (will drop later): {first_month_mask.sum():,}')

has_any_addition = df.loc[~first_month_mask, added_cols].sum(axis=1) > 0
print(f'Rows with at least one new product: {has_any_addition.sum():,} ({has_any_addition.mean():.2%})')

Computing additions: 100%|██████████| 24/24 [00:01<00:00, 12.94it/s]


First-month rows (will drop later): 956,645
Rows with at least one new product: 447,903 (3.53%)


## 3. Feature Engineering

**LEAKAGE PREVENTION:** All product-based features use lag-1 (previous month) values.
Current month product columns are ONLY used to compute the target (added_cols).

In [7]:
# --- 3A: DEMOGRAPHIC FEATURES ---

df['age_bin'] = pd.cut(df['age'], bins=[0, 25, 35, 45, 55, 65, 100],
                       labels=['18-25', '26-35', '36-45', '46-55', '56-65', '65+'])
df['log_renta'] = np.log1p(df['renta'])
df['renta_bin'] = pd.qcut(df['renta'], q=5, labels=['very_low', 'low', 'medium', 'high', 'very_high'], duplicates='drop')
df['tenure_bin'] = pd.cut(df['antiguedad'], bins=[-1, 12, 36, 72, 120, 999],
                          labels=['<1yr', '1-3yr', '3-6yr', '6-10yr', '10yr+'])
df['is_new_customer'] = (df['antiguedad'] <= 6).astype('int8')
df['join_month'] = df['fecha_alta'].dt.month.astype('float32')
df['join_year'] = df['fecha_alta'].dt.year.astype('float32')
df['months_since_join'] = ((df['fecha_dato'] - df['fecha_alta']).dt.days / 30.44).clip(lower=0)

print('Demographic features created.')

Demographic features created.


In [8]:
# --- 3B: LAG FEATURES (previous month product ownership) ---
# MUST be computed BEFORE portfolio features

for col in tqdm(product_cols, desc='Creating lag-1 features'):
    df[f'{col}_lag1'] = df.groupby('ncodpers')[col].shift(1).fillna(0).astype('int8')

lag1_cols = [f'{c}_lag1' for c in product_cols]
print('Lag-1 features created.')

Creating lag-1 features: 100%|██████████| 24/24 [00:02<00:00, 11.65it/s]

Lag-1 features created.


In [9]:
# --- 3C: PRODUCT PORTFOLIO FEATURES (from LAG-1 only — NO LEAKAGE) ---
# CRITICAL: All portfolio metrics use PREVIOUS month's products.

df['total_products'] = df[lag1_cols].sum(axis=1).astype('int8')
df['product_diversity'] = (df['total_products'] / len(product_cols)).astype('float32')
df['has_any_product'] = (df['total_products'] > 0).astype('int8')

df['payroll_bundle'] = (
    df['ind_nomina_ult1_lag1'] + df['ind_nom_pens_ult1_lag1'] + df['ind_recibo_ult1_lag1']
).astype('int8')

df['has_current_account'] = df['ind_cco_fin_ult1_lag1'].astype('int8')
df['has_credit_card'] = df['ind_tjcr_fin_ult1_lag1'].astype('int8')

account_lag_cols = [f'{c}_lag1' for c in [
    'ind_cco_fin_ult1', 'ind_cno_fin_ult1', 'ind_ctju_fin_ult1',
    'ind_ctma_fin_ult1', 'ind_ctop_fin_ult1', 'ind_ctpp_fin_ult1',
    'ind_ecue_fin_ult1', 'ind_ahor_fin_ult1']]
df['n_accounts'] = df[account_lag_cols].sum(axis=1).astype('int8')

invest_lag_cols = [f'{c}_lag1' for c in [
    'ind_dela_fin_ult1', 'ind_deme_fin_ult1', 'ind_deco_fin_ult1',
    'ind_fond_fin_ult1', 'ind_valo_fin_ult1']]
df['n_investments'] = df[invest_lag_cols].sum(axis=1).astype('int8')

lending_lag_cols = [f'{c}_lag1' for c in [
    'ind_hip_fin_ult1', 'ind_pres_fin_ult1', 'ind_tjcr_fin_ult1']]
df['n_lending'] = df[lending_lag_cols].sum(axis=1).astype('int8')

print('Portfolio features created from LAG-1 (no leakage).')

Portfolio features created from LAG-1 (no leakage).


In [10]:
# --- 3D: VELOCITY & CHANGE FEATURES ---

df['total_products_lag1'] = df['total_products'].copy()

# Lag-2 for velocity computation (comparing two past months)
for col in tqdm(product_cols, desc='Creating lag-2 features'):
    df[f'{col}_lag2'] = df.groupby('ncodpers')[col].shift(2).fillna(0).astype('int8')

lag2_cols = [f'{c}_lag2' for c in product_cols]

# Products gained/lost between lag-2 and lag-1 (both past = safe)
df['products_gained'] = (df[lag1_cols].values - df[lag2_cols].values).clip(min=0).sum(axis=1).astype('int8')
df['products_lost'] = (df[lag2_cols].values - df[lag1_cols].values).clip(min=0).sum(axis=1).astype('int8')
df['net_product_change'] = (df['products_gained'] - df['products_lost']).astype('int8')

# Rolling mean of total products (all lag-based)
df['total_products_rolling3'] = (
    df.groupby('ncodpers')['total_products']
    .transform(lambda x: x.rolling(3, min_periods=1).mean())
).astype('float32')

df['product_momentum'] = (df['total_products'] - df['total_products_rolling3']).astype('float32')

df['activity_lag1'] = df.groupby('ncodpers')['ind_actividad_cliente'].shift(1).fillna(0).astype('float32')
df['activity_change'] = (df['ind_actividad_cliente'] - df['activity_lag1']).astype('float32')

# Drop lag-2 columns (only needed for velocity)
df.drop(columns=lag2_cols, inplace=True)

print('Velocity features created.')

Creating lag-2 features: 100%|██████████| 24/24 [00:02<00:00, 11.55it/s]


Velocity features created.


In [11]:
# --- 3E: TEMPORAL FEATURES ---

df['month'] = df['fecha_dato'].dt.month.astype('int8')
df['year'] = df['fecha_dato'].dt.year.astype('int16')
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12).astype('float32')
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12).astype('float32')
df['is_q1'] = (df['month'] <= 3).astype('int8')
df['is_post_summer'] = df['month'].isin([9, 10]).astype('int8')

print('Temporal features created.')

Temporal features created.


In [12]:
# --- 3F: ENCODE CATEGORICALS ---
import re

df['is_male'] = (df['sexo'] == 'H').astype('int8')

segment_dummies = pd.get_dummies(df['segmento'], prefix='seg', dtype='int8')
df = pd.concat([df, segment_dummies], axis=1)

canal_str = df['canal_entrada'].astype(str)
top_channels = canal_str.value_counts().head(10).index
df['canal_clean'] = canal_str.where(canal_str.isin(top_channels), 'OTHER')
canal_dummies = pd.get_dummies(df['canal_clean'], prefix='canal', dtype='int8')
df = pd.concat([df, canal_dummies], axis=1)

df['is_spain'] = (df['pais_residencia'] == 'ES').astype('int8')

prov_str = df['nomprov'].astype(str)
top_provs = prov_str.value_counts().head(10).index
df['prov_clean'] = prov_str.where(prov_str.isin(top_provs), 'OTHER')
prov_dummies = pd.get_dummies(df['prov_clean'], prefix='prov', dtype='int8')
df = pd.concat([df, prov_dummies], axis=1)

indrel_dummies = pd.get_dummies(df['indrel_1mes'], prefix='indrel1m', dtype='int8')
df = pd.concat([df, indrel_dummies], axis=1)

tiprel_dummies = pd.get_dummies(df['tiprel_1mes'], prefix='tiprel', dtype='int8')
df = pd.concat([df, tiprel_dummies], axis=1)

# Clean column names for LightGBM compatibility
clean_names = {col: re.sub(r'[^A-Za-z0-9_]', '_', col) for col in df.columns}
df.rename(columns=clean_names, inplace=True)

# Update reference lists with clean names
product_cols = [re.sub(r'[^A-Za-z0-9_]', '_', c) for c in product_cols]
added_cols = [re.sub(r'[^A-Za-z0-9_]', '_', c) for c in added_cols]
lag1_cols = [re.sub(r'[^A-Za-z0-9_]', '_', c) for c in lag1_cols]

# Remove any duplicate columns
df = df.loc[:, ~df.columns.duplicated()]

print(f'After encoding: {df.shape}')

After encoding: (13647309, 163)


## 4. Final Feature Set & Train/Val Split

In [13]:
# Remove first month per customer (no lag data)
first_month_mask = df.groupby('ncodpers').cumcount() == 0
df = df[~first_month_mask].reset_index(drop=True)
print(f'After removing first months: {df.shape}')

# --- DEFINE FEATURE SET ---
exclude_cols = set(
    ['ncodpers', 'fecha_dato', 'fecha_alta', 'sexo', 'segmento', 'canal_entrada',
     'canal_clean', 'nomprov', 'prov_clean', 'pais_residencia', 'ind_empleado',
     'indresi', 'indext', 'indfall', 'indrel_1mes', 'tiprel_1mes',
     'age_bin', 'renta_bin', 'tenure_bin', 'cod_prov']
    + product_cols   # CURRENT month products = leakage
    + added_cols      # targets
)
import re
exclude_cols = {re.sub(r'[^A-Za-z0-9_]', '_', c) for c in exclude_cols}

feature_cols = [c for c in df.columns if c not in exclude_cols]
print(f'Number of features: {len(feature_cols)}')

# Verify no leakage
leaked = [c for c in feature_cols if c in product_cols]
assert len(leaked) == 0, f'LEAKAGE DETECTED: {leaked}'
print('Leakage check PASSED: no current-month product columns in features.')

After removing first months: (12690664, 163)
Number of features: 95
Leakage check PASSED: no current-month product columns in features.


In [14]:
# --- TRAIN / VALIDATION SPLIT ---
months_sorted = sorted(df['fecha_dato'].unique())
val_month = months_sorted[-1]
train_months = months_sorted[:-1]

train_df = df[df['fecha_dato'].isin(train_months)].copy()
val_df = df[df['fecha_dato'] == val_month].copy()

print(f'Train: {train_df.shape} | months: {len(train_months)}')
print(f'Val:   {val_df.shape}   | month: {val_month}')

Train: (11763904, 163) | months: 15
Val:   (926760, 163)   | month: 2016-05-28 00:00:00


In [15]:
# --- SAVE PROCESSED DATA ---
df = df.loc[:, ~df.columns.duplicated()]
feature_cols = [c for c in df.columns if c not in exclude_cols]

import json
with open(PROCESSED_PATH / 'feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)
with open(PROCESSED_PATH / 'added_cols.json', 'w') as f:
    json.dump(added_cols, f)

train_df = df[df['fecha_dato'].isin(train_months)].copy()
val_df = df[df['fecha_dato'] == val_month].copy()

save_cols = ['ncodpers', 'fecha_dato'] + feature_cols + added_cols
train_df[save_cols].to_parquet(PROCESSED_PATH / 'train.parquet', index=False)
val_df[save_cols].to_parquet(PROCESSED_PATH / 'val.parquet', index=False)

print(f'Saved to {PROCESSED_PATH}/')
print(f'train.parquet: {train_df[save_cols].shape}')
print(f'val.parquet: {val_df[save_cols].shape}')

Saved to ../data/processed/
train.parquet: (11763904, 121)
val.parquet: (926760, 121)


In [16]:
# --- FEATURE SUMMARY ---
print('='*60)
print('FEATURE ENGINEERING SUMMARY')
print('='*60)
print(f'Total features: {len(feature_cols)}')
print(f'\nFeature groups:')
print(f'  Demographics:  age, log_renta, antiguedad, is_male, is_spain, is_new_customer')
print(f'  Portfolio:     total_products, product_diversity, payroll_bundle, n_accounts (ALL from lag-1)')
print(f'  Lag-1:         24 product lag features + total_products_lag1')
print(f'  Velocity:      products_gained/lost (lag2 vs lag1), momentum, activity_change')
print(f'  Temporal:      month_sin/cos, is_q1, is_post_summer')
print(f'  Encoded cats:  segment, channel, province, indrel, tiprel')
print(f'\nTarget: 24 binary columns indicating new product additions')
print(f'\n*** LEAKAGE STATUS: CLEAN — no current-month product data in features ***')

FEATURE ENGINEERING SUMMARY
Total features: 95

Feature groups:
  Demographics:  age, log_renta, antiguedad, is_male, is_spain, is_new_customer
  Portfolio:     total_products, product_diversity, payroll_bundle, n_accounts (ALL from lag-1)
  Lag-1:         24 product lag features + total_products_lag1
  Velocity:      products_gained/lost (lag2 vs lag1), momentum, activity_change
  Temporal:      month_sin/cos, is_q1, is_post_summer
  Encoded cats:  segment, channel, province, indrel, tiprel

Target: 24 binary columns indicating new product additions

*** LEAKAGE STATUS: CLEAN — no current-month product data in features ***
